# setting up SDK

In [ ]:
! pip install -q aixplain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 600.2/600.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00


In [ ]:
import os
os.environ["AIXPLAIN_API_KEY"] = "ff6341c017618fe1a89dd129e14cd75336fcfe95e91963dcac8df9c5ff6fdd29"
from aixplain.factories import AgentFactory,  ModelFactory
from datetime import datetime
from aixplain.factories import IndexFactory
from aixplain.modules.model.index_model import Splitter
from aixplain.enums.splitting_options import SplittingOptions
from aixplain.factories import IndexFactory, ModelFactory
from aixplain.modules.model.record import Record
import time
from aixplain.factories import IndexFactory, ModelFactory
from aixplain.factories.file_factory import FileFactory


# index


In [ ]:
pricing_kb = IndexFactory.create(
    name="aiXplain Pricing Knowledge Base v6 ",
    description="SKU catalog, pricing sheets, proposal templates, and financial models",
    embedding_model="6734c55df127847059324d9e"       # OpenAI ada-002
    )

In [ ]:
print(f"Index ID: {pricing_kb.id}")

Index ID: 697a02d345d9b32ea4ca27f0


In [ ]:
from aixplain.factories import AgentFactory

pricing_index_tool = AgentFactory.create_model_tool(model=pricing_kb.id)

In [ ]:
from aixplain.factories import IndexFactory
pricing_kb = IndexFactory.get("697a02d345d9b32ea4ca27f0")
print("Index ID:", pricing_kb.id)

Index ID: 697a02d345d9b32ea4ca27f0


## Docling parser

In [ ]:
DOCLING_MODEL_ID = "677bee6c6eb56331f9192a91"
docling_model = ModelFactory.get(DOCLING_MODEL_ID)

In [ ]:
print(f" Docling Model Loaded: {docling_model.name}")
print(f"docling model ID: {docling_model.id}")

 Docling Model Loaded: Docling Document Parser (Legacy)
docling model ID: 677bee6c6eb56331f9192a91


## Dynamic Upload Function

For each file:
   * a. Upload via FileFactory
   * b. Parse with Docling
   * c. Wrap in Record with metadata
   * d. Define Splitter
   * e. Upsert to Index

In [ ]:
def process_and_upload(file_path, metadata, split_option, split_length=1, split_overlap=0):

    file_name = os.path.basename(file_path)
    print(f"\nProcessing file: {file_name}")
    print(f"Settings: Split={split_option.value}, Length={split_length}, Overlap={split_overlap}")

    # A. Upload File
    print("Step 1: Uploading file to aiXplain server...")
    try:
        uploaded_response = FileFactory.upload(file_path)
        if isinstance(uploaded_response, str):
            file_url = uploaded_response
        elif hasattr(uploaded_response, 'url'):
            file_url = uploaded_response.url
        else:
            file_url = str(uploaded_response)

    except Exception as e:
        print(f"Upload failed: {e}")
        return

    # B. Run Docling (OCR/Parsing)
    print("Step 2: Extracting text using Docling...")
    ocr_response = docling_model.run({"data": file_url})

    if ocr_response.status != "SUCCESS":
        print(f"OCR Error: {ocr_response.error_message}")
        return

    extracted_text = ""
    if isinstance(ocr_response.data, dict):
        extracted_text = ocr_response.data.get("text", str(ocr_response.data))
    else:
        extracted_text = str(ocr_response.data)

    print(f"Text Extracted: {len(extracted_text)} chars")

    # C. Create Record (Text + Metadata)
    print("Step 3: Creating Record with ARD Metadata...")

    metadata["source_filename"] = file_name
    metadata["upload_timestamp"] = time.strftime("%Y-%m-%d")

    record_id = f"rec_{file_name.replace(' ', '_').replace('.', '_')}_{int(time.time())}"

    record = Record(
        id=record_id,
        value=extracted_text,
        value_type="text",
        attributes=metadata
    )

    # D. Configure Splitter (Dynamic Overlap)
    print(f"Step 4: Configuring Splitter...")
    splitter = Splitter(
        split=True,
        split_by=split_option,
        split_length=split_length,
        split_overlap=split_overlap
    )

    # E. Upsert
    print("Step 5: Upserting to Knowledge Base...")
    pricing_kb.upsert([record], splitter=splitter)
    print("Success. Document indexed.")
    return record_id

## EXECUTION

In [ ]:
from google.colab import files
import shutil

def upload_from_desktop_and_process(label, metadata, split_option, split_length, split_overlap):
    print(f"\n[{label}] Please upload the file from your computer:")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded.")
        return

    filename = list(uploaded.keys())[0]
    file_path = f"/content/{filename}"

    print(f"File uploaded: {filename}")

    process_and_upload(
        file_path=file_path,
        metadata=metadata,
        split_option=split_option,
        split_length=split_length,
        split_overlap=split_overlap
    )

# 1. Upload Pricing Framework
upload_from_desktop_and_process(
    label="FILE 1: Pricing Framework",
    metadata={
        "document_category": "Internal_Pricing_Model",
        "content_type": "Policy_Document",
        "effective_date": "2025-11-17",
        "access_level": "Internal_Only"
    },
    split_option=SplittingOptions.PAGE,
    split_length=1,
    split_overlap=0
)

# 2. Upload Partner Slides
upload_from_desktop_and_process(
    label="FILE 2: Partner Slides (How aiXplain Does Business)",
    metadata={
        "document_category": "Partner_Program_Docs",
        "content_type": "Presentation_Slides",
        "effective_date": "2025-11-01",
        "access_level": "Partner_Ready"
    },
    split_option=SplittingOptions.PAGE,
    split_length=1,
    split_overlap=0
)

# 3. Upload Proposal Template
upload_from_desktop_and_process(
    label="FILE 3: Proposal Template",
    metadata={
        "document_category": "Templates",
        "content_type": "Master_Template",
        "unique_id": "MASTER_PROPOSAL_V1"
    },
    split_option=SplittingOptions.SENTENCE,
    split_length=50000,
    split_overlap=0
)

print("-" * 50)
print(f" Final Document Count: {pricing_kb.count()}")


[FILE 1: Pricing Framework] Please upload the file from your computer:


Saving aiXplain's Enterprise Pricing Framework.pdf to aiXplain's Enterprise Pricing Framework (1).pdf
File uploaded: aiXplain's Enterprise Pricing Framework (1).pdf

Processing file: aiXplain's Enterprise Pricing Framework (1).pdf
Settings: Split=page, Length=1, Overlap=0
Step 1: Uploading file to aiXplain server...
Step 2: Extracting text using Docling...
Text Extracted: 22942 chars
Step 3: Creating Record with ARD Metadata...
Step 4: Configuring Splitter...
Step 5: Upserting to Knowledge Base...
Success. Document indexed.

[FILE 2: Partner Slides (How aiXplain Does Business)] Please upload the file from your computer:


Saving How aiXplain Does Business_.pdf to How aiXplain Does Business_ (2).pdf
File uploaded: How aiXplain Does Business_ (2).pdf

Processing file: How aiXplain Does Business_ (2).pdf
Settings: Split=page, Length=1, Overlap=0
Step 1: Uploading file to aiXplain server...
Step 2: Extracting text using Docling...
Text Extracted: 16527 chars
Step 3: Creating Record with ARD Metadata...
Step 4: Configuring Splitter...
Step 5: Upserting to Knowledge Base...
Success. Document indexed.

[FILE 3: Proposal Template] Please upload the file from your computer:


Saving Proposal Template.pdf to Proposal Template (1).pdf
File uploaded: Proposal Template (1).pdf

Processing file: Proposal Template (1).pdf
Settings: Split=page, Length=1, Overlap=0
Step 1: Uploading file to aiXplain server...
Step 2: Extracting text using Docling...
Text Extracted: 15076 chars
Step 3: Creating Record with ARD Metadata...
Step 4: Configuring Splitter...
Step 5: Upserting to Knowledge Base...
Success. Document indexed.
--------------------------------------------------
 Final Document Count: 3


## checking the index

In [ ]:
from aixplain.factories import IndexFactory

KB_ID = pricing_kb.id
print(f"Connecting to index: {KB_ID}")

kb = IndexFactory.get(KB_ID)

test_queries = [
    "aiXPU aiXMU difference",
    "Partner Tier Discounts Core",
    "Proposal Validity Payment"
]

print("\nTesting information retrieval...")

for query in test_queries:
    print(f"\n{'='*40}")
    print(f"Searching for: '{query}'")
    print(f"{'='*40}")

    try:
        results = kb.search(query)

        if results and hasattr(results, 'data'):
            data = results.data
            found_text = ""

            if isinstance(data, list) and len(data) > 0:
                first_match = data[0]
                found_text = first_match.get("text", "") or str(first_match)
            else:
                found_text = str(data)

            if len(found_text) > 10:
                print(f"Information found ({len(found_text)} chars)!")
                print(f"Snippet:\n...{found_text[:400]}...")
            else:
                print("Results are empty or too short.")
        else:
            print("No results found.")

    except Exception as e:
        print(f"Technical error during search: {e}")

Connecting to index: 697a02d345d9b32ea4ca27f0

Testing information retrieval...

Searching for: 'aiXPU aiXMU difference'
Information found (22942 chars)!
Snippet:
...<!-- image -->

## aiXplainʼs Enterprise Pricing Framework

## 1.  Introduction and Purpose

aiXplain is being adopted in increasingly complex enterprise environments. To support this growth, pricing must be simple to understand, modular to configure, and predictable to scale. The objective of this framework is to give both customers and internal teams a clear structure for how aiXplain is license...

Searching for: 'Partner Tier Discounts Core'
Information found (22942 chars)!
Snippet:
...<!-- image -->

## aiXplainʼs Enterprise Pricing Framework

## 1.  Introduction and Purpose

aiXplain is being adopted in increasingly complex enterprise environments. To support this growth, pricing must be simple to understand, modular to configure, and predictable to scale. The objective of this framework is to give both customers and

## historical deals

In [ ]:
import os

deals_data = [
    # 1. Banking / GCC / Private Cloud (Pattern: Security + High Price)
    {
        "id": "1001", "ind": "Banking", "reg": "GCC (KSA)", "size": "Enterprise", "dep": "Private Cloud",
        "req": "Client needed strict data residency in Riyadh and high throughput for 24/7 support.",
        "sol": ["2x aiXPU", "2x aiXMU (Large Context)", "1x CMS", "3x aiXCU-L"],
        "rat": "CMS required for KSA data residency. aiXCU-L needed for 500+ concurrent banking sessions.",
        "arr": "$650,000", "tcv": "$1,950,000 (3-year)"
    },
    # 2. Healthcare / US / Private Cloud (Pattern: HIPAA + CMS)
    {
        "id": "1002", "ind": "Healthcare", "reg": "US", "size": "Mid-Market", "dep": "Private Cloud",
        "req": "Hospital system needing HIPAA compliant patient agent.",
        "sol": ["1x aiXPU", "1x aiXMU", "1x CMS"],
        "rat": "Private Cloud with CMS is mandatory for HIPAA compliance.",
        "arr": "$120,000", "tcv": "$240,000 (2-year)"
    },
    # 3. Retail / EU / SaaS (Pattern: GDPR + Burst Traffic)
    {
        "id": "1003", "ind": "Retail", "reg": "EU (Germany)", "size": "Enterprise", "dep": "SaaS",
        "req": "E-commerce giant needing agent for Black Friday scaling. GDPR compliance required.",
        "sol": ["1x aiXPU", "1x aiXCU-L (Auto-scaling)", "5000 Credits"],
        "rat": "SaaS allowed for fast deployment. aiXCU-L handles burst traffic.",
        "arr": "$200,000", "tcv": "$200,000 (1-year)"
    },
    # 4. Government / GCC / On-Prem (Pattern: Max Security + Long Term)
    {
        "id": "1004", "ind": "Government", "reg": "GCC (UAE)", "size": "Government", "dep": "On-Premise",
        "req": "Ministry services agent. Zero internet connectivity allowed for core engine.",
        "sol": ["3x aiXPU", "1x CMS (Air-gapped)", "Custom Deployment Fee"],
        "rat": "On-Prem air-gapped setup requires CMS and specialized deployment services.",
        "arr": "$400,000", "tcv": "$2,000,000 (5-year)"
    },
    # 5. Education / US / SaaS / Standard Tier (Pattern: Low Cost)
    {
        "id": "1005", "ind": "Education", "reg": "US", "size": "Small", "dep": "SaaS",
        "req": "University chatbot for student FAQs. Budget constraint.",
        "sol": ["1x aiXPU", "1x aiXMU"],
        "rat": "Standard SaaS deployment is most cost-effective for non-sensitive data.",
        "arr": "$45,000", "tcv": "$45,000 (1-year)"
    },
    # 6. Legal / EU / Private Cloud (Pattern: Heavy Memory)
    {
        "id": "1006", "ind": "Legal Services", "reg": "EU (France)", "size": "Mid-Market", "dep": "Private Cloud",
        "req": "Law firm needing to analyze thousands of contracts.",
        "sol": ["1x aiXPU", "3x aiXMU (Memory Intensive)", "1x CMS"],
        "rat": "Multiple aiXMUs required to index massive legal archives.",
        "arr": "$150,000", "tcv": "$450,000 (3-year)"
    },
    # 7. Tech / US / Partner Nexus (Pattern: Big Discount)
    {
        "id": "1007", "ind": "Technology", "reg": "US", "size": "Enterprise", "dep": "SaaS",
        "req": "Tech partner reselling agent to their SMB base.",
        "sol": ["10x aiXPU (Bulk)", "Partner Portal Access"],
        "rat": "Nexus Partner Tier qualifies for 15% discount on bulk recurring licenses.",
        "arr": "$300,000 (Discounted)", "tcv": "$900,000"
    },
    # 8. Media / GCC / SaaS (Pattern: GenAI focus)
    {
        "id": "1008", "ind": "Media", "reg": "GCC (Qatar)", "size": "Mid-Market", "dep": "SaaS",
        "req": "News agency automating content summaries in Arabic.",
        "sol": ["1x aiXPU", "2x aiXCU-M", "Arabic Fine-tuning"],
        "rat": "aiXCU-M needed for heavy LLM inference tokens.",
        "arr": "$110,000", "tcv": "$220,000"
    },
    # 9. Manufacturing / EU / On-Prem (Pattern: IoT + Edge)
    {
        "id": "1009", "ind": "Manufacturing", "reg": "EU (Italy)", "size": "Enterprise", "dep": "On-Premise",
        "req": "Factory floor safety agent monitoring sensors.",
        "sol": ["1x aiXPU", "Edge Deployment Kit"],
        "rat": "On-prem required to reduce latency to milliseconds.",
        "arr": "$90,000", "tcv": "$270,000"
    },
    # 10. Startup / US / SaaS / Edge Tier (Pattern: Entry Level)
    {
        "id": "1010", "ind": "Startup", "reg": "US", "size": "Small", "dep": "SaaS",
        "req": "Y-Combinator startup building a travel agent.",
        "sol": ["1x aiXPU", "Pay-as-you-go Credits"],
        "rat": "Edge Tier (Startup) allows low commit with credit overages.",
        "arr": "$35,000", "tcv": "$35,000"
    },
    # 11. Logistics / GCC / Private Cloud (Pattern: Hybrid)
    {
        "id": "1011", "ind": "Logistics", "reg": "GCC (UAE)", "size": "Large", "dep": "Private Cloud",
        "req": "Shipping tracker agent integrating with legacy ERP.",
        "sol": ["1x aiXPU", "1x CMS", "Integration Services"],
        "rat": "CMS needed to bridge secure Private Cloud with legacy ERP.",
        "arr": "$130,000", "tcv": "$390,000"
    },
    # 12. Insurance / US / Private Cloud (Pattern: Audit Trails)
    {
        "id": "1012", "ind": "Insurance", "reg": "US", "size": "Enterprise", "dep": "Private Cloud",
        "req": "Claims processing agent. Requires 7-year audit logs.",
        "sol": ["2x aiXPU", "1x CMS (Audit Module enabled)", "1x aiXMU"],
        "rat": "CMS Audit logs are essential for insurance compliance.",
        "arr": "$180,000", "tcv": "$540,000"
    },
    # 13. HR Tech / EU / SaaS / Core Partner (Pattern: Reseller)
    {
        "id": "1013", "ind": "HR Tech", "reg": "EU (UK)", "size": "Mid-Market", "dep": "SaaS",
        "req": "HR platform embedding our agent.",
        "sol": ["1x aiXPU", "1x aiXMU", "White-label Add-on"],
        "rat": "Core Partner gets 12% discount. White-label allows branding.",
        "arr": "$60,000", "tcv": "$120,000"
    },
    # 14. Energy / GCC / On-Prem (Pattern: Critical Infra)
    {
        "id": "1014", "ind": "Energy", "reg": "GCC (Kuwait)", "size": "Enterprise", "dep": "On-Premise",
        "req": "Oil rig operational assistant.",
        "sol": ["1x aiXPU", "Offline Model Pack", "1x CMS"],
        "rat": "Offline capability is critical for remote rigs.",
        "arr": "$250,000", "tcv": "$1,250,000 (5-year)"
    },
    # 15. Telecom / US / Hybrid (Pattern: Scale)
    {
        "id": "1015", "ind": "Telecom", "reg": "US", "size": "Enterprise", "dep": "Private Cloud",
        "req": "Customer support for 1M+ subscribers.",
        "sol": ["5x aiXPU", "10x aiXCU-L", "2x CMS"],
        "rat": "Massive concurrency requires horizontal scaling of XPU and XCU.",
        "arr": "$800,000", "tcv": "$2,400,000"
    }
]

folder_name = "historical_deals_kb"
os.makedirs(folder_name, exist_ok=True)

for deal in deals_data:
    filename = f"{folder_name}/deal_{deal['id']}_{deal['ind']}.txt"

    sku_list = "\n".join([f"- {item}" for item in deal['sol']])

    content = f"""# Historical Deal: {deal['ind']} AI Assistant (2024/2025)

## Customer Profile
Deal ID: #{deal['id']}
Customer Industry: {deal['ind']}
Region: {deal['reg']}
Customer Size: {deal['size']}
Deployment: {deal['dep']}
Status: Closed-Won
Type: Historical Deal Reference

## User Request / Context
{deal['req']}

## Recommended Solution
{sku_list}

## Rationale (Why this was chosen?)
{deal['rat']}

## Deal Outcome
- ARR: {deal['arr']}
- TCV: {deal['tcv']}
- Renewal: Yes
"""

    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)

print(f"done : {len(deals_data)}  '{folder_name}'.")


done : 15  'historical_deals_kb'.


In [ ]:
import os
import time
from aixplain.factories import IndexFactory
from aixplain.modules.model.record import Record
from aixplain.modules.model.index_model import Splitter
from aixplain.enums.splitting_options import SplittingOptions

# 1. Connect to current KB
EXISTING_KB_ID = "697a02d345d9b32ea4ca27f0"
pricing_kb = IndexFactory.get(EXISTING_KB_ID)
print(f"Connected to KB: {EXISTING_KB_ID}")

# 2. Function to upload files without Docling
def upload_deal_direct(file_path):
    file_name = os.path.basename(file_path)
    print(f"Processing: {file_name}")

    try:
        parts = file_name.replace('.txt', '').split('_')
        deal_id = parts[1]
        industry = parts[2]

        # Read text directly from file
        with open(file_path, 'r', encoding='utf-8') as f:
            text_content = f.read()

        # Add useful metadata for recommendations
        metadata = {
            "document_type": "historical_deal",
            "content_purpose": "recommendation",
            "deal_id": deal_id,
            "industry": industry,
            "upload_date": time.strftime("%Y-%m-%d")
        }

        record_id = f"deal_{deal_id}_{int(time.time())}"
        record = Record(
            id=record_id,
            value=text_content,
            value_type="text",
            attributes=metadata
        )
        splitter = Splitter(
            split=False,
            split_length=5000,
            split_overlap=0
        )

        pricing_kb.upsert([record], splitter=splitter)
        print(f"Successfully added: {file_name}")
        return True

    except Exception as e:
        print(f"Error with {file_name}: {e}")
        return False

# 3. Execute upload for all files
folder_path = "historical_deals_kb"
if os.path.exists(folder_path):
    files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    print(f"Found {len(files)} deals. Starting direct upload...\n")

    success_count = 0
    for file_name in files:
        full_path = os.path.join(folder_path, file_name)
        if upload_deal_direct(full_path):
            success_count += 1
        time.sleep(0.5)

    print(f"\nUpload complete: {success_count}/{len(files)} deals added to KB")
    print(f"Total documents in KB: {pricing_kb.count()}")
else:
    print("Folder 'historical_deals_kb' not found!")

Connected to KB: 697a02d345d9b32ea4ca27f0
Found 15 deals. Starting direct upload...

Processing: deal_1005_Education.txt
Successfully added: deal_1005_Education.txt
Processing: deal_1014_Energy.txt
Successfully added: deal_1014_Energy.txt
Processing: deal_1003_Retail.txt
Successfully added: deal_1003_Retail.txt
Processing: deal_1006_Legal Services.txt
Successfully added: deal_1006_Legal Services.txt
Processing: deal_1008_Media.txt
Successfully added: deal_1008_Media.txt
Processing: deal_1002_Healthcare.txt
Successfully added: deal_1002_Healthcare.txt
Processing: deal_1015_Telecom.txt
Successfully added: deal_1015_Telecom.txt
Processing: deal_1013_HR Tech.txt
Successfully added: deal_1013_HR Tech.txt
Processing: deal_1010_Startup.txt
Successfully added: deal_1010_Startup.txt
Processing: deal_1009_Manufacturing.txt
Successfully added: deal_1009_Manufacturing.txt
Processing: deal_1012_Insurance.txt
Successfully added: deal_1012_Insurance.txt
Processing: deal_1007_Technology.txt
Successful

# tools

## SKU lookup tool


In [ ]:
from aixplain.factories import ModelFactory

def pricing_db_tool_framework_official(sku_names: str) -> dict:
    """
    OFFICIAL Pricing Database (Enterprise Pricing Framework - Jan 2026)
    Source: Official Framework Text Document
    Input: Comma-separated SKUs (e.g. "aiXPU, aiXCU-M")
    Returns: Price & Metadata
    """

    DATABASE = {
        "sku": [
            # Core Platform
            "aiXPU",
            "aiXMU",
            "Platform_Bundle",

            # Compute (Annual Prices from Framework)
            "aiXCU-S",
            "aiXCU-M",
            "aiXCU-L",

            # Management & Services
            "CMS",
            "aiXpert_Resident_Onsite",

            # Platform Credits (2,500-credit blocks)
            "Platform_Credits_2500",

            # Implementation Services
            "Implementation_Basic",
            "Implementation_Advanced"
        ],
        "unit_price": [
            # Core Platform (Section 3.1, 3.2, 3.3)
            35000,   # aiXPU
            10000,   # aiXMU
            45000,   # Platform Bundle

            # Compute (Section 3.4 - Annual Prices)
            264000,  # aiXCU-S (27,200 TPM)
            432000,  # aiXCU-M (67,200 TPM)
            888000,  # aiXCU-L (168,000 TPM)

            # Services (Section 3.5, 3.7)
            37500,   # CMS per node/year
            225000,  # aiXpert Resident Onsite (annual)

            # Credits (Section 3.6)
            2500,    # 2,500-credit bundle

            # Implementation (estimates - not in Framework)
            10000,   # Basic
            50000    # Advanced
        ],
        "type": [
            # Core Platform
            "recurring", "recurring", "recurring",

            # Compute
            "recurring", "recurring", "recurring",

            # Services
            "recurring", "recurring",

            # Credits
            "one-time",

            # Implementation
            "one-time", "one-time"
        ],
        "description": [
            # Core Platform
            "Agentic OS License (Annual) - Framework v2026",
            "Memory Layer 1TB / 5TB usable (Annual)",
            "Platform Bundle: aiXPU + aiXMU (Annual)",

            # Compute
            "Compute Small: 27,200 TPM (Annual)",
            "Compute Medium: 67,200 TPM (Annual)",
            "Compute Large: 168,000 TPM (Annual)",

            # Services
            "Compute Management Services (Per Node, Annual)",
            "aiXpert Resident Engineer On-Site (Annual)",

            # Credits
            "Platform Credits Bundle (2,500 credits, Annual)",

            # Implementation
            "Basic Implementation Services",
            "Advanced Implementation Services"
        ]
    }

    # Build lookup map
    lookup_map = {}
    for i, item_sku in enumerate(DATABASE["sku"]):
        lookup_map[item_sku.lower()] = {
            "sku": item_sku,
            "unit_price": DATABASE["unit_price"][i],
            "type": DATABASE["type"][i],
            "description": DATABASE["description"][i]
        }

    # Process requests
    results = {}
    requested = [s.strip().lower() for s in sku_names.split(",")]

    for req in requested:
        if req in lookup_map:
            results[lookup_map[req]["sku"]] = lookup_map[req]
        else:
            # Fuzzy matching
            found = False
            for key, val in lookup_map.items():
                if req in key or key in req:
                    results[val["sku"]] = val
                    found = True
                    break
            if not found:
                results[req] = {"error": "SKU Not Found in Official Database"}

    return results

print("Deploying OFFICIAL Lookup Tool (Framework v2026)...")

try:
    lookup_framework = ModelFactory.create_utility_model(
        name="SKU Lookup Framework",
        description="Official Pricing Database (Enterprise Framework Jan 2026) - Source of Truth for all quotes.",
        code=pricing_db_tool_framework_official
    )

    lookup_framework.deploy()
    print(" Lookup Tool Deployed Successfully!")
    print(f" Tool ID: {lookup_framework.id}")

    LOOKUP_TOOL_ID = lookup_framework.id
    print(f"SAVED: LOOKUP_TOOL_ID = {LOOKUP_TOOL_ID}")

except Exception as e:
    print(f" Deployment Failed: {e}")

Deploying OFFICIAL Lookup Tool (Framework v2026)...


/tmp/ipython-input-1365623258.py:129: DeprecationWarning: create_utility_model is deprecated. Please use create_script_connection_tool instead.
  lookup_framework = ModelFactory.create_utility_model(
/usr/local/lib/python3.12/dist-packages/aixplain/modules/model/utility_model.py:243: UserWarning: WARNING: Non-deployed utility models (status=DRAFT) will expire after 24 hours after creation. Use .deploy() method to make the model permanent.
  warnings.warn(
ERROR:root:Utility Model Update Error: {'message': 'Error: Utility name already exists', 'error': 'Unprocessable Entity', 'statusCode': 422}


 Deployment Failed: Error deploying because of backend error: Utility Model Update Error: {'message': 'Error: Utility name already exists', 'error': 'Unprocessable Entity', 'statusCode': 422}


## Calculation TOOL

* Perform accurate financial calculations (ARR, TCV, 3-year TCO)

In [ ]:
from aixplain.factories import ModelFactory
import json
import ast

def pricing_calculator_v2_final(input_payload: str) -> dict:
    import json

    try:
        if isinstance(input_payload, str):
            try:
                data = json.loads(input_payload)
            except:
                data = json.loads(input_payload.replace("'", '"'))
        else:
            data = input_payload

        if data is None:
            return {"status": "error", "message": "Input data is None"}

        items = data.get("items", [])
        region = data.get("region", "US")
        partner_tier = data.get("partner_tier", "Standard")
        years = data.get("contract_years", 1)

        if not items:
            return {"status": "error", "message": "No items provided"}

        MULTIPLIERS = {"US": 1.0, "EU": 1.15, "GCC": 1.30}
        DISCOUNTS = {"Standard": 0, "Edge": 0, "Core": 12, "Nexus": 15}

        recur_subtotal = 0
        once_total = 0
        errors = []

        region_mult = MULTIPLIERS.get(region, 1.0)
        partner_disc_pct = DISCOUNTS.get(partner_tier, 0)

        for i, item in enumerate(items):
            qty = item.get("qty") or item.get("quantity", 1)
            unit_price = item.get("unit_price") or item.get("price", 0)
            sku = item.get("sku", "Unknown SKU")
            item_type = item.get("type", "recurring").lower()

            if unit_price == 0:
                errors.append(f"Item {i+1} ({sku}): Missing unit_price")
                continue

            if item_type == "recurring":
                final_unit_price = unit_price * region_mult
            else:
                final_unit_price = unit_price

            line_total = final_unit_price * qty

            if item_type == "recurring":
                recur_subtotal += line_total
            else:
                once_total += line_total

        discount_amount = recur_subtotal * (partner_disc_pct / 100)
        arr = recur_subtotal - discount_amount
        tcv = (arr * years) + once_total

        result = {
            "status": "success",
            "currency": "USD",
            "arr": round(arr, 2),
            "tcv": round(tcv, 2),
            "breakdown": {
                "region_applied": region,
                "region_multiplier": region_mult,
                "gross_recurring": round(recur_subtotal, 2),
                "partner_tier": partner_tier,
                "discount_percentage": f"{partner_disc_pct}%",
                "discount_amount": round(discount_amount, 2),
                "one_time_fees": round(once_total, 2),
                "contract_years": years
            }
        }

        if errors:
            result["warnings"] = errors

        return result

    except Exception as e:
        return {"status": "error", "message": str(e)}


print("="*70)
print("CREATING NEW SCRIPT TOOL (Auto-Deployed)")
print("="*70)

calc_tool = ModelFactory.create_script_connection_tool(
    name="Pricing Calc V7 Final Fix",
    description="Official Calculator V2 via Script Tool",
    code=pricing_calculator_v2_final
)

print(f" Created Successfully!")
print(f" Tool Name: {calc_tool.name}")
print(f" Tool ID: {calc_tool.id}")
print("="*70)

CALCULATOR_TOOL_ID = calc_tool.id

In [ ]:
CALCULATOR_TOOL_ID = "69830a4d7545a7b07139c04e"
print(f" SAVED: CALCULATOR_TOOL_ID = {CALCULATOR_TOOL_ID}")

 SAVED: CALCULATOR_TOOL_ID = 69830a4d7545a7b07139c04e


#  agent

## agent insturctions and creation

In [ ]:
from aixplain.factories import AgentFactory, ModelFactory, IndexFactory

OPTIMIZED_INSTRUCTIONS = """
AGENT IDENTITY & ROLE

You are aiXplain's Pricing Specialist Agent — an internal pricing authority with comprehensive knowledge of aiXplain's SKUs, pricing policies, proposal templates, commercial strategy, and deal structures.

Your primary users are internal teams: Solutions, Sales, Product, Finance, and Executive Leadership.


CORE RESPONSIBILITIES

1. PRICING INFORMATION
   - Provide instant, accurate answers on aiXplain's pricing components
   - Explain SKU differences, dependencies, and use cases
   - Act as the single source of truth for pricing rules and calculation logic

2. COMMERCIAL PROPOSALS
   - Generate full commercial proposals using aiXplain's official template
   - Maintain consistent formatting, tone, and SKU naming conventions
   - Include: introduction, assumptions, pricing table, ARR/TCV, optional add-ons

3. FINANCIAL MODELING
   - Translate technical solution designs into financial structures
   - Calculate ARR (Annual Recurring Revenue), TCV (Total Contract Value), and multi-year TCO
   - Distinguish between recurring vs one-time revenue correctly

4. PRICING STRATEGY ADVICE
   - Advise on pricing strategy, discounting ranges, and commercial models
   - Tailor recommendations based on customer size, region, and deal type
   - Compare subscription vs perpetual models when relevant

------

### ⚠️ CRITICAL: ENFORCED MISSING INFORMATION PROTOCOL
Before processing ANY query or calling ANY tool, you MUST perform this exact verification process:

REQUIRED PARAMETERS FOR PRICING:
1. SKUs and quantities
2. Region (MUST be one of: US, EU, GCC)
3. Partner tier (MUST be one of: Standard, Edge, Core, Nexus)
4. Contract length (MUST be in years: 1, 2, or 3)

VERIFICATION PROCEDURE (MANDATORY):
1. Scan user input for ALL four required parameters
2. If ANY single parameter is missing:
   - DO NOT proceed with calculations
   - DO NOT use default values
   - IMMEDIATELY ask user for SPECIFICALLY which parameters are missing
   - Format the request clearly: "Before I can provide pricing, I need to know: [missing parameters]"
3. ONLY after ALL parameters are explicitly provided may you proceed

EXAMPLES:

Input: "I need pricing for 2 aiXPUs"
Response: "Before I can provide pricing, I need to know:
- Region (US, EU, or GCC)
- Partner tier (Standard, Edge, Core, or Nexus)
- Contract length (1, 2, or 3 years)"

Input: "Quote for 1 aiXPU in US region with Core partner"
Response: "Before I can provide pricing, I need to know:
- Contract length (1, 2, or 3 years)"

TOOL 1: SKU Lookup Framework

Purpose: Retrieve official unit prices from the pricing database
Input: Comma-separated SKU names (e.g., "aiXPU, aiXCU-M, CMS")
Output: Price, type (recurring/one-time), and description per SKU

MANDATORY: Always call this tool FIRST to get unit_price before any
calculation. Never assume or invent prices.


TOOL 2: Pricing Calculator

Purpose: Perform accurate ARR, TCV, and TCO calculations

INPUT FORMAT (EXACT JSON STRUCTURE REQUIRED):
{
  "input_payload": "{
    \"items\": [
      {\"sku\": \"aiXPU\", \"unit_price\": 35000, \"qty\": 2,
       \"type\": \"recurring\"}
    ],
    \"region\": \"US\",
    \"partner_tier\": \"Standard\",
    \"contract_years\": 3
  }"
}

REQUIRED FIELDS FOR EACH ITEM:
- sku: SKU name
- unit_price: MUST come from Lookup Tool (never guess)
- qty: quantity
- type: "recurring" OR "one-time"

REGIONAL MULTIPLIERS (applied automatically):
- US: 1.0x (base)
- EU: 1.15x
- GCC: 1.30x

PARTNER TIER DISCOUNTS (applied to recurring only):
- Standard: 0%
- Edge: 0%
- Core: 12%
- Nexus: 15%


CALCULATOR USAGE - ABSOLUTE REQUIREMENT

CRITICAL: You MUST use the Calculator Tool for ALL pricing calculations - ZERO EXCEPTIONS !

Even for simple math like "2 × $35,000" or "multiply by 3 years", ALWAYS use the Calculator Tool.

FORBIDDEN: Never perform any pricing calculations manually, regardless of how simple they seem.

When ANY calculation is needed:
1. Get prices from Lookup Tool
2. ALWAYS call Calculator Tool with proper JSON format
3. Return the exact results from Calculator Tool

TOOL 3: aiXplain Pricing Knowledge Base v6

MANDATORY INPUT FORMAT:
{
  "data": "<YOUR SEARCH QUERY HERE>",
  "dataType": "text"
}

CORRECT EXAMPLES:
✓ {"data": "How many concurrent agents can aiXPU handle?",
   "dataType": "text"}

✓ {"data": "Core partner tier annual fee",
   "dataType": "text"}

WRONG:
✗ search("query") ← Missing JSON!
✗ {"query": "..."} ← Wrong key!

-----------------------------------------------------------------------

CRITICAL: PROPOSAL TEMPLATE RETRIEVAL
When generating proposals, use this EXACT query:
{"data": "unique_id:MASTER_PROPOSAL_V1", "dataType": "text"}

This retrieves the COMPLETE 7-page template as ONE document.
DO NOT search for "proposal template" - it may return fragments.

###  MANDATORY TOOL USAGE - ZERO EXCEPTIONS

FORBIDDEN BEHAVIOR:
-  Creating proposal structure without fetching template from KB !
-  Calculating pricing without the Calculator tool
-  Making up prices without using Lookup tool

REQUIRED WORKFLOW (ENFORCED):
1. FETCH TEMPLATE: KB Tool MUST return template_text
2. LOOKUP PRICES: Get accurate pricing from Lookup Tool
3. CALCULATE: Calculator Tool MUST compute ARR/TCV
4. FILL TEMPLATE YOURSELF: You must fill in the template with all values


DECISION-MAKING FRAMEWORK

When a user makes a request, follow this process:

STEP 1: CLARIFY THE OBJECTIVE
Determine if the user wants:
□ A pricing breakdown
□ A commercial proposal
□ A comparison between models
□ A SKU recommendation
□ Financial modeling (ARR, TCV, TCO)
□ Policy/concept explanation

STEP 2: GATHER NECESSARY INPUTS
If any are missing, ask follow-up questions:
□ Customer profile (size: Small/Mid/Large, region, industry)
□ Deployment type (SaaS, Private Cloud, On-Prem)
□ Required agentic capabilities
□ Expected consumption/concurrency level
□ Contract length (1, 2, or 3 years)
□ Partner involvement and tier

STEP 3: APPLY CORRECT PRICING LOGIC
□ Call Lookup Tool for unit prices
□ Identify all required SKUs including dependencies
□ Adjust for customer tier and region
□ Apply partner discounts if applicable

STEP 4: PRODUCE STRUCTURED OUTPUT
□ Clean, consistent language
□ Pricing table in aiXplain format
□ ARR, TCV, and multi-year TCO calculations
□ Clear assumptions stated
□ Footnotes where needed

STEP 5: CONFIRM ALIGNMENT
□ Explain your reasoning
□ Highlight any assumptions made
□ Suggest optional add-ons if relevant


RECOMMENDATION FROM HISTORICAL DEALS:

When recommending SKUs based on customer requirements:

1. ALWAYS search for similar historical deals FIRST using this EXACT format:
   {"data": "document_type:historical_deal [CUSTOMER REQUIREMENTS]", "dataType": "text"}

   Examples:
   ✓ {"data": "document_type:historical_deal private cloud high concurrency", "dataType": "text"}
   ✓ {"data": "document_type:historical_deal healthcare HIPAA", "dataType": "text"}
   ✓ {"data": "document_type:historical_deal [INDUSTRY] [REGION] [DEPLOYMENT]", "dataType": "text"}

2. After analyzing historical deals, identify patterns:
   - Which SKUs appear together for similar requirements?
   - What dependencies exist (e.g., CMS always included with Private Cloud)?
   - How does SKU selection change based on industry, size, or region?

3. Base your recommendation on these historical patterns first, then apply standard rules.

EXAMPLE: Recommendation Based on Historical Deals
─────────────────────────────────────
User: "Client wants to build an HR agent on private cloud with high concurrency"

Correct Workflow:
1. Search historical deals:
   {"data": "document_type:historical_deal HR private cloud high concurrency", "dataType": "text"}

2. Analyze returned deals (e.g., find HR Tech deals or similar industries with private cloud)

3. Identify common patterns:
   - All private cloud deployments include CMS
   - HR solutions typically require aiXMU
   - High concurrency requires aiXCU-M or aiXCU-L

4. Recommend SKUs based on historical patterns:
   "Based on our historical deals for similar requirements, I recommend:
    - 1x aiXPU (required for orchestration)
    - 2x aiXMU (for HR knowledge context)
    - 1x aiXCU-M (for high concurrency)
    - 1x CMS (required for private cloud deployment)"

5. Continue with standard pricing workflow using these SKUs


WORKFLOW EXAMPLES

EXAMPLE 1: Simple Pricing Calculation
─────────────────────────────────────
User: "Calculate pricing for 2 aiXPUs for a 3-year contract in the US"

Correct Workflow:
1. Call Lookup("aiXPU") → unit_price = $35,000
2. Build items: [{"sku": "aiXPU", "unit_price": 35000, "qty": 2, "type": "recurring"}]
3. Call Calculator with: region="US", partner_tier="Standard", contract_years=3
4. Present results: ARR = $70,000 | TCV = $210,000

WRONG: Manually calculating 2 × $35,000 × 3 = $210,000


EXAMPLE 2: Complex SKU Recommendation
─────────────────────────────────────
User: "Client wants an HR agent on private cloud with high concurrency"

Correct Workflow:
1. Identify requirements: Private Cloud + High Concurrency
2. Required SKUs (with dependencies):
   - aiXPU (base orchestration) ✓
   - aiXMU (memory for HR context) ✓
   - aiXCU-M (high concurrency compute) ✓
   - CMS (MANDATORY for private cloud!) ✓
3. Call Lookup for all SKUs to get prices
4. Call Calculator with complete items list
5. Present: SKU list + prices + rationale + total ARR/TCV

WRONG: Recommending only aiXPU + aiXMU (missing CMS and compute!)


EXAMPLE 3: Multi-Region Quote
─────────────────────────────
User: "Quote for 1 aiXPU, 1 aiXMU, 1 aiXCU-M in GCC region, Core partner, 2 years"

Correct Workflow:
1. Lookup all SKUs → aiXPU=$35K, aiXMU=$10K, aiXCU-M=$432K
2. Build items array with all three SKUs
3. Call Calculator with: region="GCC", partner_tier="Core", contract_years=2
4. Calculator applies: GCC 1.30x multiplier, Core 12% discount
5. Present breakdown showing: gross, regional adjustment, discount, final ARR/TCV


PROPOSAL WORKFLOW - MANUAL FILLING (STRICT MODE)

STEP 1: Gather Requirements & Calculate
   - Call Lookup Tool for prices.
   - Call Calculator Tool for ARR/TCV.

STEP 2: FETCH TEMPLATE
   - Call KB Tool: {"data": "unique_id:MASTER_PROPOSAL_V1", "dataType": "text"}
   - This retrieves the "XYZ" template.

STEP 3: FILL THE TEMPLATE (DO NOT REWRITE IT)
   - You must output the EXACT structure and text of the retrieved template.
   - Preserve all Section Numbers (1., 2.1, 2.2, etc.) exactly as they appear.
   - Preserve all legal text and definitions exactly as they appear.
   - ONLY replace placeholders (e.g., [Client Name], [Date]) and populate the Pricing Tables with the calculated data.
   - Do NOT summarize the Executive Summary; just replace the client name and details.
   - Do NOT change the section titles.

STEP 4: OUTPUT GENERATION
   - Output the full document.
   - Use Markdown only to preserve the bolding and headers of the original document.

### CRITICAL:
If the template has "2.1 Agentic OS Platform License", your output MUST have "2.1 Agentic OS Platform License".
Do not invent new section headers like "Recommended Solution".
Stick to the template's table of contents structure.

###  QUALITY CONTROL CHECKLIST
Before submitting ANY proposal, verify:
□ Did I fetch the template using KB tool?
□ Did I look up all prices using Lookup Tool?
□ Did I calculate pricing using Calculator tool?
□ Did I fill ALL placeholders in the template?
□ Are my tables properly formatted in markdown?

If ANY answer is NO, fix the proposal before submitting.


CONSTRAINTS & BOUNDARIES

ABSOLUTE PROHIBITIONS — NEVER DO THESE:

✗ Approve discounts outside approved ranges (Standard/Edge/Core/Nexus)
✗ Invent pricing outside aiXplain's actual SKU catalog
✗ Override legal or commercial policies
✗ Send anything directly to customers without human review
✗ Approve transactions, contractual commitments, or legal terms
✗ Modify SKU costs or alter aiXplain's pricing logic
✗ Make assumptions when information is missing — ASK instead
✗ Calculate manually — ALWAYS use the Calculator tool

ERROR HANDLING:
- If Calculator fails: verify input format, retry once with correct structure
- If still failing: inform user "Calculator error — please contact support"
- Never enter infinite retry loops
- Never fabricate results if tools fail

OUTPUT FORMAT & TONE

TONE:
- Internal, professional, and precise
- Never overly formal or robotic
- Clear step-by-step reasoning when appropriate
- Prioritize correctness over speculation

PROPOSAL OUTPUT FORMAT:
When generating commercial proposals, include:
1. Introduction/Executive Summary
2. Customer Requirements & Assumptions
3. Recommended Solution (SKUs with rationale)
4. Pricing Table (SKU | Qty | Unit Price | Line Total | Type)
5. Financial Summary (ARR, TCV, multi-year TCO if applicable)
6. Optional Add-ons & Recommendations
7. Terms & Validity Notes

CALCULATION OUTPUT FORMAT:
Always present calculations as:

PRICING SUMMARY
Annual Recurring Revenue (ARR): $X
Total Contract Value (TCV): $Y
Contract Term: Z years

BREAKDOWN
- Region: [region] (multiplier)
- Partner Tier: [tier] (discount)
- One-time Fees: $X


PERFORMANCE TARGETS

- Response time: < 30 seconds for interactive queries
- Accuracy: Zero calculation errors; all prices from official sources
- Completeness: All SKU dependencies included
- Compliance: 100% adherence to pricing policies
"""

LLM_ID = "669a63646eb56306647e1091" # GPT-4o
llm = ModelFactory.get(LLM_ID)
lookup_tool = ModelFactory.get("6979e5ac3668e41b13a55afc")
calc_tool = ModelFactory.get("69830c62c69015a15b22dce4")
kb_tool = IndexFactory.get("697a02d345d9b32ea4ca27f0")

tools_list = [
    lookup_tool,
    calc_tool,
    kb_tool,
]

try:
    corrected_agent = AgentFactory.create(
        name="pricing agent v6",
        description="Official Pricing Agent with Ultimate Proposal Generator",
        instructions=OPTIMIZED_INSTRUCTIONS,
        llm=llm,
        tools=tools_list,
    )
    print(f" Agent Created! ID: {corrected_agent.id}")
    AGENT_ID = corrected_agent.id

except Exception as e:
    print(f" Error: {e}")

except Exception as e:
    print(f" Error: {e}")

 Agent Created! ID: 69860eceff9be055415d5765


## Testing the agent proformance

In [ ]:
import uuid

agent = AgentFactory.get(AGENT_ID)

chat_session_id = f"{agent.id}_chat_{str(uuid.uuid4())[:8]}"

print("enter you question .\n")

while True:
    user_input = input("\n You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("conversation ended")
        break
    if not user_input.strip():
        continue
    try:
        response = agent.run(user_input, session_id=chat_session_id)

        print(f" Agent: {response.data.output}")

    except Exception as e:
        print(f" Error: {e}")

enter you question .


 You: I need a quote for an enterprise customer who wants 2 aiXPUs and 1 aiXCU-M for their banking solution.
🤖  pricing agent v6 | ⏳ Initialization
⚙️  pricing agent v6 | GPT-4o Mini | ⏳
🤖  pricing agent v6 | ⏳ Formatting Output
🤖  pricing agent v6 | ✓ Formatting Output
✅ Done | (7.7 s total | 2 API calls | $0.00191385)
 Agent: Before I can provide pricing, I need to know: 
- Region (US, EU, or GCC) 
- Partner tier (Standard, Edge, Core, or Nexus) 
- Contract length (1, 2, or 3 years)


In [22]:
import time
import json
from aixplain.factories import AgentFactory

# Load Agent
agent = AgentFactory.get(AGENT_ID)

test_results = []

def run_test(test_name, query, expected_behavior, max_time=10):
    """Run a single test and track performance."""
    print(f"\n{'='*70}")
    print(f"{'='*70}")
    print(f"📝 Query: {query}")
    print(f"🎯 Expected: {expected_behavior}")

    start_time = time.time()

    try:
        # Run agent
        response = agent.run(query)
        elapsed = time.time() - start_time

        # Extract answer
        if hasattr(response, 'data'):
            raw_data = response.data
            if hasattr(raw_data, 'output'):
                answer = str(raw_data.output)
            elif hasattr(raw_data, 'content'):
                answer = str(raw_data.content)
            else:
                answer = str(raw_data)
        else:
            answer = str(response)

        # Performance check
        passed = elapsed < max_time

        result = {

            "full_response": answer,
            "response_preview": answer[:300] + "..." if len(answer) > 300 else answer
        }


        print(f"📄 Full Response:")
        print(answer)
        print()

        test_results.append(result)
        return result

    except Exception as e:
        elapsed = time.time() - start_time
        print(f" ERROR: {e}")
        test_results.append({
            "test": test_name,
            "status": " FAIL",
            "error": str(e),
            "time": round(elapsed, 2)
        })
        return None

# Run tests
run_test(
    test_name="1. Explain Pricing Model",
    query="Can you explain how aiXPU and aiXMU differ? What are they used for?",
    expected_behavior="Accurate explanation from KB",
    max_time=8
)

run_test(
    test_name="2. Recommend SKUs",
    query="Client wants to build an HR agent on private cloud with high concurrency. What SKUs do they need?",
    expected_behavior="Recommends aiXPU, aiXMU, CMS",
    max_time=10
)

run_test(
    test_name="3. Calculate ARR/TCV",
    query="Calculate pricing for 2 aiXPUs for a 3-year contract in the US with Standard partner tier.",
    expected_behavior="Returns ARR=$70K, TCV=$210K",
    max_time=10
)

run_test(
    test_name="5. Pricing Strategy",
    query="Should a large enterprise customer in UAE go with subscription or perpetual licensing?",
    expected_behavior="Recommends subscription model",
    max_time=8
)

run_test(
    test_name="6. Dependency Validation",
    query="Can a customer buy aiXMU without aiXPU?",
    expected_behavior="Says NO, explains dependency",
    max_time=6
)




📝 Query: Can you explain how aiXPU and aiXMU differ? What are they used for?
🎯 Expected: Accurate explanation from KB
🤖  pricing agent v6 | ⏳ Initialization
⚙️  pricing agent v6 | GPT-4o Mini | ⏳
⚙️  pricing agent v6 | aiXplain Pricing Knowledge Base v6 | ⏳
⚙️  pricing agent v6 | GPT-4o Mini | ⏳
🤖  pricing agent v6 | ⏳ Formatting Output
✅ Done | (17.1 s total | 4 API calls | $0.0061988999999999985)
📄 Full Response:
aiXPU (aiXplain Processing Unit) is the control layer of the aiXplain platform. It handles agent orchestration, workflow execution, access control, and governance. It is essential for any agentic deployment, whether hosted on-premise or in a private cloud. aiXPU is required for managing agent execution and ensuring compliance with enterprise policies.

On the other hand, aiXMU (aiXplain Memory Unit) is the memory layer that stores and retrieves knowledge used by agents for reasoning, recall, and grounding. It can be shared across environments or segmented for security or pe

{'full_response': 'No, a customer cannot buy aiXMU without aiXPU. aiXPU must be present to orchestrate access and retrieval.',
 'response_preview': 'No, a customer cannot buy aiXMU without aiXPU. aiXPU must be present to orchestrate access and retrieval.'}